# Algorithm21 Tutorial Audit — KLDM-PPR-lite with Wyckoff Projection and CrystalFormer Ranking

This notebook is the staged implementation and audit notebook for the tutorial pipeline:

- `Algorithm21A`: oracle-space-group / oracle-template, q-only Wyckoff projection
- `Algorithm21B`: fixed-template CrystalFormer q-sample-rank, top-3 branch renoise
- `Algorithm21C`: oracle-space-group template proposal diagnostics with PyXtal/CrystalFormer

It also cross-checks the design against local source implementations in `src/PPR` and `src/CrystalFormer`.


## CrystalFormer weights

Recommended reference links:
- [CrystalFormer available weights](https://github.com/deepmodeling/CrystalFormer#available-weights)
- [CrystalFormer CSP Colab](https://colab.research.google.com/drive/1I7b5exbB2oBjexFIEaeDQexmYRDgLHVk?authuser=0#scrollTo=kfu6Ez9e6Sp7)

The CSP-only checkpoint expected by this notebook is:
- `epoch_046000.pkl`

You can either:
- set `KLDM_ALGO21_CF_CHECKPOINT` to a local checkpoint path
- or run the download cell below


## CrystalFormer dependency bootstrap

**Purpose:** CrystalFormer needs a small JAX-side stack before the checkpoint wrapper can load. If the preflight below reports missing packages such as `haiku`, run the optional install cell once, restart the kernel, and rerun from the top.


In [2]:
# Optional dependency bootstrap for CrystalFormer.
# Run this only if the dependency preflight reports missing packages.

# %pip install -U "jax[cpu]"
# %pip install dm-haiku==0.0.15 optax==0.2.6 pyxtal==1.1.1 gdown


In [3]:
import importlib.util
import pandas as pd
rows = []
for module_name in ('jax', 'haiku', 'optax', 'pyxtal', 'gdown'):
    rows.append({
        'test': 'algorithm21_cf_dependency_preflight',
        'module': module_name,
        'available': bool(importlib.util.find_spec(module_name) is not None),
    })

dep_df = pd.DataFrame(rows)
display(dep_df)


,test,module,available
0,algorithm21_cf_dependency_preflight,jax,True
1,algorithm21_cf_dependency_preflight,haiku,True
2,algorithm21_cf_dependency_preflight,optax,True
3,algorithm21_cf_dependency_preflight,pyxtal,True
4,algorithm21_cf_dependency_preflight,gdown,True


In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("JAX_DISABLE_JIT", "true")
os.environ.setdefault("KLDM_ALGO21_SAFE_MODE", "true")
import time
import math
import traceback
import subprocess
import sys
import pickle
import json as jsonlib
import importlib
import gc
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from pymatgen.core import Element

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm19_kldm_ppr_diffcsppp as algo19_backend
import kldmPlus.algorithm21_clean_cf_ppr_kldm as algo21_backend
algo19_backend = importlib.reload(algo19_backend)
algo21_backend = importlib.reload(algo21_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case
from kldmPlus.utils.time import iter_sampling_times, make_times
from kldmPlus.symmetry.wyckoff_templates import extract_wyckoff_templates, flatten_site_signature, sample_random_free_vars, recover_template_free_vars_from_anchor_entries

Algorithm19State = algo19_backend.Algorithm19State
build_oracle_diffcsppp_payload_from_structure = algo19_backend.build_oracle_diffcsppp_payload_from_structure
map_model_to_payload_reference_chart = algo19_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo19_backend.map_payload_reference_chart_to_model_frame
kldm_clean_fractional_denoiser_Df = algo19_backend.kldm_clean_fractional_denoiser_Df
wrap01 = algo19_backend.wrap01
wrapdiff = algo19_backend.wrapdiff
torus_rmse = algo19_backend.torus_rmse

Algorithm21Config = algo21_backend.Algorithm21Config
CrystalFormerLikelihood = algo21_backend.CrystalFormerLikelihood
q_only_clean_cf_fit = algo21_backend.q_only_clean_cf_fit
q_only_clean_cf_local_rerank = algo21_backend.q_only_clean_cf_local_rerank
algorithm21_project_renoise_step = algo21_backend.algorithm21_project_renoise_step
algorithm21_project_renoise_step_from_q = algo21_backend.algorithm21_project_renoise_step_from_q
algorithm21_project_renoise_step_local_rerank = algo21_backend.algorithm21_project_renoise_step_local_rerank
algorithm21b_project_renoise_step = algo21_backend.algorithm21b_project_renoise_step
algorithm21_should_project = algo21_backend.algorithm21_should_project
predict_clean_f0 = algo21_backend.predict_clean_f0
model_to_payload = algo21_backend.model_to_payload
payload_to_model = algo21_backend.payload_to_model
expand_q = algo21_backend.expand_q
fit_q_to_clean_prediction = algo21_backend.fit_q_to_clean_prediction
torus_soft_project = algo21_backend.torus_soft_project
renoise_from_f0 = algo21_backend.renoise_from_f0
post_renoise_score = algo21_backend.post_renoise_score
post_renoise_acceptance = algo21_backend.post_renoise_acceptance
sample_q_from_crystalformer = algo21_backend.sample_q_from_crystalformer
rank_q_candidates = algo21_backend.rank_q_candidates
build_payload_from_template_q = algo21_backend.build_payload_from_template_q
species_match_reorder = algo21_backend.species_match_reorder
build_crystalformer_reduced_sequence = algo21_backend.build_crystalformer_reduced_sequence
expand_crystalformer_reduced_sequence = algo21_backend.expand_crystalformer_reduced_sequence
crystalformer_reduced_sequence_debug = algo21_backend.crystalformer_reduced_sequence_debug
crystalformer_site_representative_search = algo21_backend.crystalformer_site_representative_search
crystalformer_payload_order_assembly_debug = algo21_backend.crystalformer_payload_order_assembly_debug
torus_interp = algo21_backend.torus_interp

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO21_PROFILE', 'laptop').strip().lower()
def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def notebook_log(message: str) -> None:
    text = str(message)
    print(text)
    with ALGO21_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(text)
        if not text.endswith("\n"):
            handle.write("\n")

def cf_nll_eval(*, payload, q, lattice_feature, formula, label: str) -> float:
    notebook_log(f"[cf-eval] start {label}")
    if cf_like is None:
        notebook_log(f"[cf-eval] skip {label} cf_like unavailable")
        return float('nan')
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'false')).strip().lower() in {'1', 'true', 'yes', 'on'}
    try:
        q_np = np.asarray(torch.as_tensor(q).detach().cpu(), dtype=float)
        l_np = np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float)
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
            in_path = tmp_dir / f'{safe_label}.pkl'
            out_path = tmp_dir / f'{safe_label}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'q': q_np,
                    'lattice_feature': l_np,
                    'formula': formula,
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm21_cf_score_once.py'), '--input', str(in_path), '--output', str(out_path)]
            notebook_log(f"[cf-eval] subprocess start {label}")
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '240')))
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                notebook_log(f"[cf-eval] subprocess ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
                raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer subprocess failed')
            value = float(result['value'])
        else:
            notebook_log(f"[cf-eval] in-kernel start {label}")
            value = float(cf_like.nll_q(
                payload=payload,
                q=q_np,
                lattice_feature=l_np,
                formula=formula,
            ))
            notebook_log(f"[cf-eval] in-kernel done {label}")
        notebook_log(f"[cf-eval] done {label} value={value:.6g}")
        return value
    except Exception as exc:
        notebook_log(f"[cf-eval] ERROR {label} {type(exc).__name__}: {exc}")
        raise
    finally:
        notebook_cleanup()

def cf_sequence_smoke(*, payload, q, lattice_feature, label: str) -> dict[str, Any]:
    notebook_log(f"[cf-seq] start {label}")
    seq = build_crystalformer_reduced_sequence(payload=payload, q=q, lattice_feature=lattice_feature)
    notebook_log(f"[cf-seq] done {label} num_sites={int(seq['num_sites'])}")
    notebook_cleanup()
    return seq

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO21_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO21_GRAPH_IDS', '2', '2'))
CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_CORE_T', '0.1,0.3,0.5', '0.1,0.3,0.5')))
BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_BETA_VALUES', '0,0.001,0.01,0.1,1,10', '0,0.001,0.01,0.1,1,10')))
ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_ALPHA_VALUES', '0,0.1,0.25,0.5,1.0', '0,0.1,0.25,0.5,1.0')))
Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_Q_OPT_STEPS', '50', '50'))
TEST45_BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_BETA_VALUES', '0', '0')))
TEST6_ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST6_ALPHA_VALUES', '0,0.25', '0,0.25')))
TEST46_Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_TEST46_Q_OPT_STEPS', '4', '4'))
TEST46_CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST46_CORE_T', '0.5', '0.5')))
TEST7_RERANK_RADIUS = float(profile_default('KLDM_ALGO21_TEST7_RERANK_RADIUS', '0.05', '0.05'))
TEST7_RERANK_CANDIDATES = int(profile_default('KLDM_ALGO21_TEST7_RERANK_CANDIDATES', '3', '3'))
TEST7_RERANK_KEEP_TOL = float(profile_default('KLDM_ALGO21_TEST7_RERANK_KEEP_TOL', '0.05', '0.05'))
TEST45_CF_SCALE_BETAS = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_CF_SCALE_BETAS', '0,1e-6,1e-5,1e-4', '0,1e-6,1e-5,1e-4')))
# CF-gradient optimization is deliberately opt-in; default notebook tests use value-only reranking.
os.environ.setdefault('KLDM_ALGO21_ENABLE_CF_GRAD', 'false')
SHORT_SAMPLER_STEPS = int(profile_default('KLDM_ALGO21_SHORT_SAMPLER_STEPS', '100', '100'))
SHORT_SAMPLER_TSTART = float(profile_default('KLDM_ALGO21_SHORT_SAMPLER_TSTART', '0.5', '0.5'))
NOTEBOOK_SAFE_MODE = str(os.environ.get("KLDM_ALGO21_SAFE_MODE", "true")).strip().lower() in {"1", "true", "yes", "on"}
SHOW_FULL_DEBUG_TABLES = not NOTEBOOK_SAFE_MODE
CF_RANDOM_TRIALS = int(profile_default('KLDM_ALGO21_CF_RANDOM_TRIALS', '1' if NOTEBOOK_SAFE_MODE else '8', '1'))
CF_CHECKPOINT = str(profile_default('KLDM_ALGO21_CF_CHECKPOINT', str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl')), str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl'))))
CF_FORMULA = os.environ.get('KLDM_ALGO21_FORMULA', '').strip() or None
ALGO21_LOG_PATH = ROOT / 'notebooks' / 'log.txt'
os.environ["KLDM_ALGO21_LOG_PATH"] = str(ALGO21_LOG_PATH)
ALGO21_LOG_PATH.write_text("", encoding="utf-8")

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

print(f'profile={PROFILE} graphs={GRAPH_IDS} t={CORE_T_VALUES} beta={BETA_VALUES} alpha={ALPHA_VALUES} q_opt_steps={Q_OPT_STEPS} cf_ckpt={CF_CHECKPOINT}')


def notebook_cleanup():
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'false')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        try:
            cf_obj = globals().get('cf_like', None)
            if cf_obj is not None and hasattr(cf_obj, 'release_runtime'):
                cf_obj.release_runtime()
        except Exception:
            pass
    gc.collect()
    try:
        import jax
        jax.clear_caches()
    except Exception:
        pass
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


In [ ]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    from collections import Counter
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    return SimpleNamespace(
        pos=pos,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
    )


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(vanilla_structure=structure, standardization='refined', symprec=1e-2, angle_tolerance=5.0)
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    return {'match': bool(result.match), 'valid': bool(result.valid), 'rmse': result.rmse, 'frac_rmse': result.frac_rmse}


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State, *, variant: str = 'minus', coordinate_score_mode: str = 'direct') -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=variant,
        coordinate_score_mode=coordinate_score_mode,
    )


def sample_gt_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None):
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    if chart.total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((int(chart.total_dof),), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q mode {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor):
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def sample_template_noisy_state(case: GraphCase, *, init_mode: str, t_value: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    set_seed(int(seed) + 70000 * int(case.graph_id) + int(round(1000.0 * float(t_value))) + (0 if init_mode == 'crystalformer_oracle' else 1))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, payload, q0, z0_payload


def q_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    if a.numel() == 0 and b.numel() == 0:
        return 0.0
    diff = wrapdiff(a.reshape(-1), b.reshape(-1))
    return float(torch.sqrt(diff.square().mean()).detach().item())


def composition_to_formula_string(composition: dict[int, int]) -> str:
    counts = [int(v) for v in composition.values() if int(v) > 0]
    gcd_value = 0
    for value in counts:
        gcd_value = int(value) if gcd_value == 0 else math.gcd(int(gcd_value), int(value))
    gcd_value = gcd_value or 1
    parts = []
    for atomic_number in sorted(int(k) for k in composition):
        count = int(composition[atomic_number])
        if count <= 0:
            continue
        reduced = count // gcd_value
        symbol = Element.from_Z(int(atomic_number)).symbol
        parts.append(symbol if reduced == 1 else f'{symbol}{reduced}')
    return ''.join(parts)


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def config21(*, beta: float, alpha: float, q_opt_steps: int = Q_OPT_STEPS, post_renoise_acceptance: bool = True, debug_prints: bool = False, cf_grad_max_dims: int | None = None, cf_value_only_after_renoise: bool = False) -> Algorithm21Config:
    return Algorithm21Config(
        beta=float(beta),
        alpha=float(alpha),
        q_opt_steps=int(q_opt_steps),
        post_renoise_acceptance=bool(post_renoise_acceptance),
        debug_prints=bool(debug_prints),
        cf_grad_max_dims=cf_grad_max_dims,
        cf_value_only_after_renoise=bool(cf_value_only_after_renoise),
    )


def fit_clean_q(clean_f: torch.Tensor, *, case: GraphCase, payload, config: Algorithm21Config, cf_like=None, formula: str | None = None, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    return q_only_clean_cf_fit(
        z_payload=z_payload,
        payload=payload,
        t_nodes=torch.full((int(clean_f.shape[0]),), 0.3, device=clean_f.device, dtype=clean_f.dtype),
        lattice_feature=case.gt_l_feature,
        formula=formula,
        config=config,
        cf_likelihood=cf_like,
        q_init=q_init,
    )


loaded_graphs= [2] sg= [4]


## Step-by-Step Audit Map

This notebook is organized as a staged audit against the tutorial. It also records which local source files were used to verify the mathematical intent:

- `src/PPR/ppr-main/ppr/projection.py`: noisy-space Predict-Project-Renoise reference
- `src/PPR/ppr-main/ppr/diffusion/samplers.py`: predict then project reverse-step structure
- `src/CrystalFormer/CrystalFormer-main/crystalformer/src/sample.py`: autoregressive `(W, A, X, Y, Z, L)` sampling factorization
- `src/CrystalFormer/CrystalFormer-main/crystalformer/src/loss.py`: teacher-forced log-probability decomposition used for `q` scoring


In [ ]:
audit_rows = [
    {'tutorial_part': 'A1-A6', 'backend_status': 'implemented', 'notebook_coverage': 'payload sanity, q fit, init ablation, alpha sweep, renoise, short sampler'},
    {'tutorial_part': 'B1-B5', 'backend_status': 'implemented', 'notebook_coverage': 'CF q sampling, geometry rank, top-3 branches, renoise survival, short sampler'},
    {'tutorial_part': 'C1-C5', 'backend_status': 'diagnostic-first', 'notebook_coverage': 'template proposal validity, oracle-template recovery, template-q witness ranking, approximate branch projection'},
]
source_rows = [
    {'source': 'src/PPR/ppr-main/ppr/projection.py', 'verified_point': 'PPR core loop is predict/project/renoise in noisy space; Algorithm21 remains a deliberate clean-space approximation.'},
    {'source': 'src/PPR/ppr-main/ppr/diffusion/samplers.py', 'verified_point': 'Projection is inserted into the reverse step after prediction; our KLDM variant mirrors the sequencing while simplifying the optimizer.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/sample.py', 'verified_point': 'CrystalFormer samples reduced-site coordinates autoregressively with fixed W/A teacher forcing available for template-conditioned q proposals.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/loss.py', 'verified_point': 'CrystalFormer NLL decomposes into logp_g/logp_w/logp_xyz/logp_a/logp_l; Algorithm21 uses coordinate NLL as a tie-break, not as the symmetry constraint.'},
]
display(pd.DataFrame(audit_rows))
display(pd.DataFrame(source_rows))
wrapper_names = ['predict_clean_f0','model_to_payload','payload_to_model','expand_q','fit_q_to_clean_prediction','torus_soft_project','renoise_from_f0','post_renoise_acceptance','sample_q_from_crystalformer','rank_q_candidates','algorithm21b_project_renoise_step','build_payload_from_template_q']
display(pd.DataFrame({'backend_symbol': wrapper_names, 'available': [hasattr(algo21_backend, name) for name in wrapper_names]}))


,tutorial_part,backend_status,notebook_coverage
0,A1-A6,implemented,"payload sanity, q fit, init ablation, alpha sw..."
1,B1-B5,implemented,"CF q sampling, geometry rank, top-3 branches, ..."
2,C1-C5,diagnostic-first,"template proposal validity, oracle-template re..."


,source,verified_point
0,src/PPR/ppr-main/ppr/projection.py,PPR core loop is predict/project/renoise in no...
1,src/PPR/ppr-main/ppr/diffusion/samplers.py,Projection is inserted into the reverse step a...
2,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer samples reduced-site coordinates...
3,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer NLL decomposes into logp_g/logp_...


,backend_symbol,available
0,predict_clean_f0,True
1,model_to_payload,True
2,payload_to_model,True
3,expand_q,True
4,fit_q_to_clean_prediction,True
5,torus_soft_project,True
6,renoise_from_f0,True
7,post_renoise_acceptance,True
8,sample_q_from_crystalformer,True
9,rank_q_candidates,True


## Runtime Preflight

Load the KLDM comparison runner and, when available, the CrystalFormer checkpoint wrapper used by `Algorithm21B` and the Version C diagnostics.


In [ ]:
rows = []
cf_like = None
notebook_log('[test7] cf checkpoint load start')
try:
    ckpt_exists = Path(CF_CHECKPOINT).exists()
    cf_like = CrystalFormerLikelihood(checkpoint_path=CF_CHECKPOINT) if ckpt_exists else None
    rows.append({
        'test': 'algorithm21_cf_checkpoint_load',
        'checkpoint_path': CF_CHECKPOINT,
        'checkpoint_exists': bool(ckpt_exists),
        'wrapper_ready': bool(cf_like is not None),
        'coordinate_only': bool(cf_like.coordinate_only) if cf_like is not None else np.nan,
        'PASS': bool(cf_like is not None),
        'status': 'INFO' if cf_like is not None else 'MISSING_CHECKPOINT',
    })
except ModuleNotFoundError as exc:
    rows.append({
        'test': 'algorithm21_cf_checkpoint_load',
        'graph': 'na',
        'PASS': False,
        'status': 'MISSING_DEPENDENCY',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': f'{type(exc).__name__}: {exc}',
        'failure_category': 'cf_checkpoint_load',
        'checkpoint_path': CF_CHECKPOINT,
        'install_hint': 'Run the CrystalFormer dependency bootstrap cell, restart the kernel, and rerun.',
    })
except Exception as exc:
    rows.append(error_row('algorithm21_cf_checkpoint_load', 'na', exc, 'cf_checkpoint_load', checkpoint_path=CF_CHECKPOINT))

notebook_log(f'[test7] cf checkpoint load done wrapper_ready={bool(cf_like is not None)}')
df = dataframe_result('algorithm21_cf_checkpoint_load', rows)
notebook_log('[test7] display start')
safe_display_sorted(df, ['checkpoint_path'])
notebook_log('[test7] display done')

notebook_log('[test7] cleanup start')
notebook_cleanup()
notebook_log('[test7] cleanup done')


[test7] cf checkpoint load start
[trace] CrystalFormerLikelihood.__post_init__ start checkpoint=/workspace/notebooks/epoch_046000.pkl
[trace] CrystalFormerLikelihood._load_runtime start checkpoint=/workspace/notebooks/epoch_046000.pkl
[trace] CrystalFormerLikelihood._load_runtime step=ensure_importable
[trace] CrystalFormerLikelihood._load_runtime step=import_crystalformer
[trace] CrystalFormerLikelihood._load_runtime step=import_jax


[trace] CrystalFormerLikelihood._load_runtime step=import_cf_modules
[trace] CrystalFormerLikelihood._load_runtime step=make_prng
[trace] CrystalFormerLikelihood._load_runtime step=make_transformer
[trace] CrystalFormerLikelihood._load_runtime step=make_loss_fn
[trace] CrystalFormerLikelihood._load_runtime step=find_ckpt_filename
[trace] CrystalFormerLikelihood._load_runtime step=load_data path=/workspace/notebooks/epoch_046000.pkl
[trace] CrystalFormerLikelihood._load_runtime step=normalize_params
[trace] CrystalFormerLikelihood._load_runtime step=merge_params
[trace] CrystalFormerLikelihood._load_runtime step=jit_logp_fn
[crystalformer] init checkpoint=/workspace/notebooks/epoch_046000.pkl coordinate_only=True safe_mode=True
[test7] cf checkpoint load done wrapper_ready=True
[test7] display start


,test,checkpoint_path,checkpoint_exists,wrapper_ready,coordinate_only,PASS,status
0,algorithm21_cf_checkpoint_load,/workspace/notebooks/epoch_046000.pkl,True,True,True,True,INFO


[test7] display done
[test7] cleanup start
[test7] cleanup done


## Part A0 — Frame, Denoiser, and Local Feasibility Checks

Before the main Algorithm21A tests, verify the pieces we rely on:

- clean denoiser output is stable under the wrapper
- payload/model frame transport is self-consistent
- oracle `q_GT` really decodes to the oracle payload
- local witness around `q_GT` is well-scaled


In [ ]:
rows_a0 = []
tutorial_a0_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    try:
        state = sample_gt_noisy_state(case, t_value=tutorial_a0_t)
        f0_direct = clean_prediction(state)
        f0_wrap = predict_clean_f0(state=state, model=runner.model)
        z_gt = expand_q(payload=payload, q=q_gt)
        f_gt_model = payload_to_model(z_payload=z_gt, payload=payload)
        z_roundtrip = model_to_payload(f_model=f_gt_model, payload=payload)
        q_pert = torch.remainder(q_gt + 0.05 * torch.randn_like(q_gt), 1.0) if q_gt.numel() else q_gt.clone()
        z_pert = expand_q(payload=payload, q=q_pert)
        rows_a0.append({
            'test': 'algorithm21a_a0_frame_denoiser_feasibility',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'clean_wrapper_rmse': float(torus_rmse(f0_direct, f0_wrap).detach().item()),
            'payload_roundtrip_rmse': float(torus_rmse(z_gt, z_roundtrip).detach().item()),
            'oracle_payload_rmse_to_gt': float(torus_rmse(z_gt, torch.as_tensor(payload.expanded_frac_coords, device=z_gt.device, dtype=z_gt.dtype)).detach().item()),
            'local_witness_qgt_to_qpert': float(algo21_backend.witness_torus_sin_loss(z_gt, z_pert).detach().item()),
            'PASS': True,
            'status': 'INFO',
        })
    except Exception as exc:
        rows_a0.append(error_row('algorithm21a_a0_frame_denoiser_feasibility', case.graph_id, exc, 'frame_denoiser_feasibility', space_group=case.requested_sg))
    finally:
        notebook_cleanup()
display(dataframe_result('algorithm21a_a0_frame_denoiser_feasibility', rows_a0).sort_values(['graph']).reset_index(drop=True))


,test,graph,space_group,clean_wrapper_rmse,payload_roundtrip_rmse,oracle_payload_rmse_to_gt,local_witness_qgt_to_qpert,PASS,status
0,algorithm21a_a0_frame_denoiser_feasibility,2,4,0.0,9.618658e-09,1.773593e-08,0.03583,True,INFO


## Part A — Oracle Template / q-only Wyckoff Projection

This section implements the staged `Algorithm21A` audit only:

- `A1`: payload and Wyckoff chart sanity
- `A2`: pure q projection at fixed time
- `A3`: q initialization ablation
- `A4-A5`: alpha sweep and renoise survival


In [ ]:
rows_a1 = []
rows_a2 = []
rows_a3 = []
rows_a45 = []
tutorial_a_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
notebook_log(f'[part-a] start graphs={[case.graph_id for case in GRAPH_CASES]} t={tutorial_a_t}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    try:
        dbg = crystalformer_payload_order_assembly_debug(payload=payload, q=q_gt)
        rows_a1.append({
            'test': 'algorithm21a_a1_payload_chart_sanity',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'num_atoms': int(payload.num_atoms),
            'num_sites': int(len(payload.wyckoff_letters)),
            'payload_order_rmse': float(dbg['payload_order_rmse']),
            'global_aligned_rmse': float(dbg['global_aligned_rmse']),
            'species_order_ok': bool(sorted(payload.expanded_atomic_numbers.tolist()) == sorted(case.atomic_numbers.detach().cpu().tolist())),
            'wyckoff_letters_ok': bool(len(payload.wyckoff_letters) == len(payload.anchor_atomic_numbers)),
            'active_dof_mask_ok': bool(payload.anchor_free_coordinate_masks is not None and payload.anchor_free_coordinate_masks.shape[0] == len(payload.wyckoff_letters)),
            'PASS': bool(float(dbg['payload_order_rmse']) < 1e-5 and float(dbg['global_aligned_rmse']) < 1e-5),
            'status': 'INFO',
        })
    except Exception as exc:
        rows_a1.append(error_row('algorithm21a_a1_payload_chart_sanity', case.graph_id, exc, 'payload_chart_sanity', space_group=case.requested_sg))

    try:
        state = sample_gt_noisy_state(case, t_value=tutorial_a_t)
        f0_hat = clean_prediction(state)
        z_hat = model_to_payload(f_model=f0_hat, payload=payload)
        fit_random = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=None, q_init_mode='random', steps=TEST46_Q_OPT_STEPS)
        z_gt = expand_q(payload=payload, q=q_gt.to(device=z_hat.device, dtype=z_hat.dtype))
        witness_before = float(algo21_backend.witness_torus_sin_loss(z_gt, z_hat).detach().item())
        before_eval = evaluate_arrays(case, f0_hat, case.gt_l_feature, case.atomic_numbers)
        f0_hard = payload_to_model(z_payload=fit_random.z_proj_payload, payload=payload)
        after_eval = evaluate_arrays(case, f0_hard, case.gt_l_feature, case.atomic_numbers)
        rows_a2.append({
            'test': 'algorithm21a_a2_q_projection',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            't': float(tutorial_a_t),
            'witness_before': float(witness_before),
            'witness_after': float(fit_random.witness_sin),
            'frac_rmse_before': float(before_eval['frac_rmse']),
            'frac_rmse_after': float(after_eval['frac_rmse']),
            'q_distance_to_GT': float(q_distance(fit_random.q_star, q_gt)),
            'PASS': bool(float(fit_random.witness_sin) < float(witness_before) and float(after_eval['frac_rmse']) <= float(before_eval['frac_rmse']) + 1e-8),
            'status': 'INFO',
        })

        init_trials = []
        fit_oracle = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=q_gt, q_init_mode='oracle_structure', steps=TEST46_Q_OPT_STEPS)
        init_trials.append(('oracle_q_init', fit_oracle))
        fit_prev = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=q_gt.detach().clone(), q_init_mode='random', steps=TEST46_Q_OPT_STEPS)
        init_trials.append(('previous_live_q', fit_prev))
        fit_rand_single = fit_random
        init_trials.append(('random_q_init', fit_rand_single))
        best_multistart = None
        for multistart_idx in range(4):
            fit_try = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=None, q_init_mode='random', steps=TEST46_Q_OPT_STEPS)
            if best_multistart is None or float(fit_try.witness_sin) < float(best_multistart.witness_sin):
                best_multistart = fit_try
        init_trials.append(('multi_start_random', best_multistart))
        for init_name, fit_item in init_trials:
            rows_a3.append({
                'test': 'algorithm21a_a3_q_init_ablation',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't': float(tutorial_a_t),
                'init_mode': init_name,
                'witness_after': float(fit_item.witness_sin),
                'q_distance_to_GT': float(q_distance(fit_item.q_star, q_gt)),
                'PASS': bool(np.isfinite(float(fit_item.witness_sin))),
                'status': 'INFO',
            })

        q_use = fit_random.q_star.detach().clone()
        z_proj = expand_q(payload=payload, q=q_use.to(device=z_hat.device, dtype=z_hat.dtype))
        f0_hard = payload_to_model(z_payload=z_proj, payload=payload)
        for alpha in TEST6_ALPHA_VALUES:
            f0_star = torus_soft_project(f0_hat=f0_hat, f0_hard=f0_hard, alpha=float(alpha))
            clean_eval = evaluate_arrays(case, f0_star, case.gt_l_feature, case.atomic_numbers)
            candidate_state = renoise_from_f0(f0_star=f0_star, state=state, model=runner.model)
            accepted, fit_before_post, fit_after_post = post_renoise_acceptance(state_before=state, state_candidate=candidate_state, payload=payload, model=runner.model, q_init_before=q_use, q_init_after=q_use)
            clean_after = clean_prediction(candidate_state)
            redenoise_eval = evaluate_arrays(case, clean_after, case.gt_l_feature, case.atomic_numbers)
            rows_a45.append({
                'test': 'algorithm21a_a4_a5_alpha_renoise',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                't': float(tutorial_a_t),
                'alpha': float(alpha),
                'witness_before': float(fit_before_post.witness_sin),
                'witness_after_clean': float(algo21_backend.witness_torus_sin_loss(model_to_payload(f_model=f0_star, payload=payload), z_proj).detach().item()),
                'witness_after_redenoise': float(fit_after_post.witness_sin),
                'frac_rmse_before': float(before_eval['frac_rmse']),
                'frac_rmse_after_clean': float(clean_eval['frac_rmse']),
                'frac_rmse_after_redenoise': float(redenoise_eval['frac_rmse']),
                'accepted': bool(accepted),
                'PASS': True,
                'status': 'INFO',
            })
    except Exception as exc:
        rows_a2.append(error_row('algorithm21a_a2_q_projection', case.graph_id, exc, 'q_projection', space_group=case.requested_sg, t=float(tutorial_a_t)))
    finally:
        notebook_cleanup()
display(dataframe_result('algorithm21a_a1_payload_chart_sanity', rows_a1).sort_values(['graph']).reset_index(drop=True))
display(dataframe_result('algorithm21a_a2_q_projection', rows_a2).sort_values(['graph','t']).reset_index(drop=True))
display(dataframe_result('algorithm21a_a3_q_init_ablation', rows_a3).sort_values(['graph','init_mode']).reset_index(drop=True))
display(dataframe_result('algorithm21a_a4_a5_alpha_renoise', rows_a45).sort_values(['graph','alpha']).reset_index(drop=True))


[part-a] start graphs=[2] t=0.5


,test,graph,space_group,num_atoms,num_sites,payload_order_rmse,global_aligned_rmse,species_order_ok,wyckoff_letters_ok,active_dof_mask_ok,PASS,status
0,algorithm21a_a1_payload_chart_sanity,2,4,16,8,0.0,2.466394e-08,True,True,True,True,INFO


,test,graph,space_group,t,witness_before,witness_after,frac_rmse_before,frac_rmse_after,q_distance_to_GT,PASS,status
0,algorithm21a_a2_q_projection,2,4,0.5,0.000581,0.319769,0.007674,0.173195,0.221025,False,INFO


,test,graph,space_group,t,init_mode,witness_after,q_distance_to_GT,PASS,status
0,algorithm21a_a3_q_init_ablation,2,4,0.5,multi_start_random,0.314240,0.216593,True,INFO
1,algorithm21a_a3_q_init_ablation,2,4,0.5,oracle_q_init,0.000461,0.009074,True,INFO
2,algorithm21a_a3_q_init_ablation,2,4,0.5,previous_live_q,0.000461,0.009074,True,INFO
3,algorithm21a_a3_q_init_ablation,2,4,0.5,random_q_init,0.319769,0.221025,True,INFO


,test,graph,space_group,t,alpha,witness_before,witness_after_clean,witness_after_redenoise,frac_rmse_before,frac_rmse_after_clean,frac_rmse_after_redenoise,accepted,PASS,status
0,algorithm21a_a4_a5_alpha_renoise,2,4,0.5,0.00,0.001954,0.319769,0.001787,0.007674,0.007674,0.008591,True,True,INFO
1,algorithm21a_a4_a5_alpha_renoise,2,4,0.5,0.25,0.001954,0.215645,0.001983,0.007674,0.055385,0.011285,False,True,INFO


## Part A Intermediate — Fit Backend and Anchor Diagnostics

These checks split `Algorithm21A` into smaller feasibility questions:

- how bad is the random-init basin versus oracle-init?
- does the hard anchor improve witness before renoise?
- how large is the wrapped displacement from `f0_hat` to `f0_hard`?


In [ ]:
rows_a_mid = []
tutorial_a_mid_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    state = sample_gt_noisy_state(case, t_value=tutorial_a_mid_t)
    f0_hat = clean_prediction(state)
    z_hat = model_to_payload(f_model=f0_hat, payload=payload)
    fit_oracle = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=q_gt, q_init_mode='oracle_structure', steps=TEST46_Q_OPT_STEPS)
    fit_random = fit_q_to_clean_prediction(z_hat=z_hat, payload=payload, t_nodes=state.t_nodes, lattice_feature=case.gt_l_feature, q_init=None, q_init_mode='random', steps=TEST46_Q_OPT_STEPS)
    for label, fit_item in [('oracle', fit_oracle), ('random', fit_random)]:
        f0_hard = payload_to_model(z_payload=fit_item.z_proj_payload, payload=payload)
        hard_payload = model_to_payload(f_model=f0_hard, payload=payload)
        delta = wrapdiff(f0_hard, f0_hat)
        rows_a_mid.append({
            'test': 'algorithm21a_intermediate_fit_anchor',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'init_mode': label,
            'witness_fit': float(fit_item.witness_sin),
            'payload_redecode_rmse': float(torus_rmse(hard_payload, fit_item.z_proj_payload).detach().item()),
            'anchor_shift_rms': float(torch.sqrt(delta.square().mean()).detach().item()),
            'q_distance_to_GT': float(q_distance(fit_item.q_star, q_gt)),
            'PASS': True,
            'status': 'INFO',
        })
    notebook_cleanup()
display(dataframe_result('algorithm21a_intermediate_fit_anchor', rows_a_mid).sort_values(['graph','init_mode']).reset_index(drop=True))


,test,graph,space_group,init_mode,witness_fit,payload_redecode_rmse,anchor_shift_rms,q_distance_to_GT,PASS,status
0,algorithm21a_intermediate_fit_anchor,2,4,oracle,0.000461,1.279686e-08,0.006832,0.009074,True,INFO
1,algorithm21a_intermediate_fit_anchor,2,4,random,0.319769,2.482055e-08,0.221341,0.221025,True,INFO


## CrystalFormer q-Sampling Helper

This helper runs CrystalFormer q sampling in a subprocess by default so Part B does not crash the main notebook kernel.


In [ ]:
def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
    notebook_log(f'[cf-sample] start K={int(K)} seed={int(seed)}')
    if cf_like is None:
        raise RuntimeError('CrystalFormer checkpoint not available.')
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SAMPLE_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    q_candidates = []
    cf_nll = []
    if use_subprocess:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        in_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.pkl'
        out_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.json'
        with in_path.open('wb') as handle:
            pickle.dump({
                'repo_root': str(ROOT),
                'checkpoint_path': str(CF_CHECKPOINT),
                'symmetry_payload': payload,
                'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                'formula': formula,
                'K': int(K),
                'top_p': float(top_p),
                'temperature': float(temperature),
                'seed': int(seed),
            }, handle)
        cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm21_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
        try:
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')))
        except subprocess.TimeoutExpired as exc:
            raise RuntimeError(
                f'CrystalFormer q sampling subprocess timed out after {exc.timeout}s for K={int(K)}. '
                'Increase KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT or reduce K.'
            ) from exc
        result = {}
        if out_path.exists():
            with out_path.open('r', encoding='utf-8') as handle:
                result = jsonlib.load(handle)
        if completed.returncode != 0 or not result.get('ok'):
            tail = completed.stderr[-800:] if completed.stderr else ''
            raise RuntimeError(result.get('error_message') or tail or 'CrystalFormer q sampling subprocess failed')
        q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
        cf_nll = [float(v) for v in result.get('cf_nll', [])]
    else:
        q_candidates, cf_nll = sample_q_from_crystalformer(
            payload=payload,
            lattice_feature=torch.as_tensor(lattice_feature),
            formula=formula,
            cf_likelihood=cf_like,
            K=int(K),
            top_p=float(top_p),
            temperature=float(temperature),
            seed=int(seed),
        )
    notebook_log(f'[cf-sample] done K={int(K)} n={len(q_candidates)}')
    return q_candidates, cf_nll


## Tutorial Part B — Explicit Algorithm21B CF q-sample-rank Audit

These cells evaluate the exact tutorial path for fixed-template `Algorithm21B`:

1. sample `K` `q` candidates from CrystalFormer under the oracle template
2. rank by KLDM witness to the clean prediction
3. keep top-3 branches
4. soft-project, renoise, and keep the best accepted branch


In [ ]:
if 'sample_q_candidates_eval' not in globals():
    def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
        notebook_log(f'[cf-sample] start K={int(K)} seed={int(seed)}')
        if cf_like is None:
            raise RuntimeError('CrystalFormer checkpoint not available.')
        use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SAMPLE_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
        q_candidates = []
        cf_nll = []
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            in_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.pkl'
            out_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                    'formula': formula,
                    'K': int(K),
                    'top_p': float(top_p),
                    'temperature': float(temperature),
                    'seed': int(seed),
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm21_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')))
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                tail = completed.stderr[-800:] if completed.stderr else ''
                raise RuntimeError(result.get('error_message') or tail or 'CrystalFormer q sampling subprocess failed')
            q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
            cf_nll = [float(v) for v in result.get('cf_nll', [])]
        else:
            q_candidates, cf_nll = sample_q_from_crystalformer(
                payload=payload,
                lattice_feature=torch.as_tensor(lattice_feature),
                formula=formula,
                cf_likelihood=cf_like,
                K=int(K),
                top_p=float(top_p),
                temperature=float(temperature),
                seed=int(seed),
            )
        notebook_log(f'[cf-sample] done K={int(K)} n={len(q_candidates)}')
        return q_candidates, cf_nll

rows_b_samples = []
rows_b_branches = []
tutorial_b_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
k_values = [16]
notebook_log(f'[tutorial-b] start graphs={[case.graph_id for case in GRAPH_CASES]} K={k_values} t={tutorial_b_t}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    formula = cf_formula_for_case(case)
    chart = algo19_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    state = sample_gt_noisy_state(case, t_value=tutorial_b_t)
    f0_hat = clean_prediction(state)
    z_hat = map_model_to_payload_reference_chart(f0_hat, payload)
    for K in k_values:
        try:
            if cf_like is None:
                raise RuntimeError('CrystalFormer checkpoint not available.')
            q_candidates, cf_nll = sample_q_candidates_eval(
                payload=payload,
                lattice_feature=case.gt_l_feature,
                formula=formula,
                K=int(K),
                top_p=1.0,
                temperature=1.0,
                seed=SAMPLE_SEED + int(case.graph_id) * 100 + int(K),
            )
            q_rmse = [q_distance(q_i, q_gt) for q_i in q_candidates]
            ranked = rank_q_candidates(z_hat=z_hat, payload=payload, q_candidates=q_candidates, cf_nll=cf_nll, top_k=3, epsilon_abs=1.0e-3)
            ranked = score_ranked_q_candidates_with_crystalformer(ranked=ranked, payload=payload, lattice_feature=case.gt_l_feature, formula=formula, cf_likelihood=cf_like)
            top3_min = min([q_distance(item.q, q_gt) for item in ranked], default=float('nan'))
            top10_min = min(q_rmse[:10], default=float('nan')) if q_rmse else float('nan')
            rows_b_samples.append({
                'test': 'algorithm21b_cf_sample_quality',
                'graph': case.graph_id,
                'space_group': case.requested_sg,
                'K': int(K),
                'best_sample_rmse_to_GT': float(min(q_rmse)) if q_rmse else float('nan'),
                'top3_min_rmse_to_GT': float(top3_min),
                'top10_min_rmse_to_GT': float(top10_min),
                'mean_CF_NLL_top3': float(np.mean([item.cf_nll for item in ranked])) if ranked else float('nan'),
                'best_CF_NLL_top3': float(np.min([item.cf_nll for item in ranked])) if ranked else float('nan'),
                'top1_witness': float(ranked[0].witness_sin) if ranked else float('nan'),
                'top1_rmse_to_GT': float(q_distance(ranked[0].q, q_gt)) if ranked else float('nan'),
                'PASS': bool(np.isfinite(top3_min)),
                'status': 'INFO',
            })
            for item in ranked:
                step = algorithm21_project_renoise_step_from_q(
                    state=state,
                    payload=payload,
                    model=runner.model,
                    q_star=item.q,
                    alpha=0.25,
                    post_renoise_accept=True,
                )
                endpoint_clean = evaluate_arrays(case, step.f0_star, case.gt_l_feature, case.atomic_numbers)
                endpoint_after = evaluate_arrays(case, step.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
                rows_b_branches.append({
                    'test': 'algorithm21b_top3_branch',
                    'graph': case.graph_id,
                    'space_group': case.requested_sg,
                    'K': int(K),
                    'branch_rank': int(item.rank),
                    'candidate_witness': float(item.witness_sin),
                    'candidate_CF_NLL': float(item.cf_nll),
                    'clean_frac_rmse': float(endpoint_clean['frac_rmse']),
                    'clean_witness': float(step.fit_before.witness_sin),
                    'post_renoise_witness': float(step.fit_after.witness_sin),
                    'post_renoise_frac_rmse': float(endpoint_after['frac_rmse']),
                    'accepted': bool(step.accepted),
                    'PASS': bool(step.accepted),
                    'status': 'INFO',
                })
        except Exception as exc:
            rows_b_samples.append(error_row('algorithm21b_cf_sample_quality', case.graph_id, exc, 'algorithm21b_sampling', space_group=case.requested_sg, K=int(K)))
        finally:
            notebook_cleanup()
df_b_samples = dataframe_result('algorithm21b_cf_sample_quality', rows_b_samples)
if len(df_b_samples) and {'graph', 'K'}.issubset(df_b_samples.columns):
    df_b_samples = df_b_samples.sort_values(['graph', 'K']).reset_index(drop=True)
display(df_b_samples)

df_b_branches = dataframe_result('algorithm21b_top3_branch', rows_b_branches)
if len(df_b_branches) and {'graph', 'K', 'branch_rank'}.issubset(df_b_branches.columns):
    df_b_branches = df_b_branches.sort_values(['graph', 'K', 'branch_rank']).reset_index(drop=True)
display(df_b_branches)


[tutorial-b] start graphs=[2] K=[16, 64, 128] t=0.5
[trace] CrystalFormerLikelihood._ensure_runtime_loaded start
[trace] CrystalFormerLikelihood._composition_vector start sg=4 formula=True


## Part B Intermediate — Candidate and Branch Feasibility

Before the short sampler, inspect whether `Algorithm21B` is feasible step by step:

- are sampled `q` values finite and correctly shaped?
- does KLDM witness ranking separate the pool?
- does the best branch already help in clean space before renoise?


In [ ]:
if 'sample_q_candidates_eval' not in globals():
    def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
        notebook_log(f'[cf-sample] start K={int(K)} seed={int(seed)}')
        if cf_like is None:
            raise RuntimeError('CrystalFormer checkpoint not available.')
        use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SAMPLE_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
        q_candidates = []
        cf_nll = []
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            in_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.pkl'
            out_path = tmp_dir / f'sample_q_K{int(K)}_seed{int(seed)}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                    'formula': formula,
                    'K': int(K),
                    'top_p': float(top_p),
                    'temperature': float(temperature),
                    'seed': int(seed),
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm21_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')))
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                tail = completed.stderr[-800:] if completed.stderr else ''
                raise RuntimeError(result.get('error_message') or tail or 'CrystalFormer q sampling subprocess failed')
            q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
            cf_nll = [float(v) for v in result.get('cf_nll', [])]
        else:
            q_candidates, cf_nll = sample_q_from_crystalformer(
                payload=payload,
                lattice_feature=torch.as_tensor(lattice_feature),
                formula=formula,
                cf_likelihood=cf_like,
                K=int(K),
                top_p=float(top_p),
                temperature=float(temperature),
                seed=int(seed),
            )
        notebook_log(f'[cf-sample] done K={int(K)} n={len(q_candidates)}')
        return q_candidates, cf_nll

rows_b_mid = []
tutorial_b_mid_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
K_mid = 64
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    formula = cf_formula_for_case(case)
    state = sample_gt_noisy_state(case, t_value=tutorial_b_mid_t)
    f0_hat = clean_prediction(state)
    z_hat = model_to_payload(f_model=f0_hat, payload=payload)
    try:
        q_candidates, cf_nll = sample_q_candidates_eval(payload=payload, lattice_feature=case.gt_l_feature, formula=formula, K=K_mid, top_p=1.0, temperature=1.0, seed=SAMPLE_SEED + int(case.graph_id) * 123)
        ranked = rank_q_candidates(z_hat=z_hat, payload=payload, q_candidates=q_candidates, cf_nll=cf_nll, top_k=3, epsilon_abs=1.0e-3)
        ranked = score_ranked_q_candidates_with_crystalformer(ranked=ranked, payload=payload, lattice_feature=case.gt_l_feature, formula=formula, cf_likelihood=cf_like)
        witness_vals = [float(algo21_backend.witness_torus_sin_loss(expand_q(payload=payload, q=q_i.to(z_hat.device, z_hat.dtype)), z_hat).detach().item()) for q_i in q_candidates]
        best_clean = None
        for item in ranked:
            step = algorithm21_project_renoise_step_from_q(state=state, payload=payload, model=runner.model, q_star=item.q, alpha=0.25, post_renoise_accept=False)
            clean_eval = evaluate_arrays(case, step.f0_star, case.gt_l_feature, case.atomic_numbers)
            if best_clean is None or float(clean_eval['frac_rmse']) < float(best_clean['frac_rmse']):
                best_clean = {'frac_rmse': clean_eval['frac_rmse'], 'witness': item.witness_sin}
        rows_b_mid.append({
            'test': 'algorithm21b_intermediate_candidate_branch',
            'graph': case.graph_id,
            'space_group': case.requested_sg,
            'K': int(K_mid),
            'num_candidates': int(len(q_candidates)),
            'q_dim': int(q_candidates[0].numel()) if q_candidates else 0,
            'witness_min': float(np.min(witness_vals)) if witness_vals else float('nan'),
            'witness_median': float(np.median(witness_vals)) if witness_vals else float('nan'),
            'witness_max': float(np.max(witness_vals)) if witness_vals else float('nan'),
            'cf_nll_top3_min': float(np.min([item.cf_nll for item in ranked])) if ranked else float('nan'),
            'cf_nll_top3_median': float(np.median([item.cf_nll for item in ranked])) if ranked else float('nan'),
            'top3_best_clean_frac_rmse': float(best_clean['frac_rmse']) if best_clean is not None else float('nan'),
            'top3_best_clean_witness': float(best_clean['witness']) if best_clean is not None else float('nan'),
            'PASS': bool(len(q_candidates) == K_mid and len(ranked) > 0),
            'status': 'INFO',
        })
    except Exception as exc:
        rows_b_mid.append(error_row('algorithm21b_intermediate_candidate_branch', case.graph_id, exc, 'algorithm21b_intermediate', space_group=case.requested_sg, K=int(K_mid)))
    finally:
        notebook_cleanup()
display(dataframe_result('algorithm21b_intermediate_candidate_branch', rows_b_mid).sort_values(['graph']).reset_index(drop=True))


## Short Sampler — Baseline vs Algorithm21A vs Algorithm21B

Run a short reverse chain and compare the final witness / RMSE for:

- `baseline`
- `beta0_q_projection` (`Algorithm21A`)
- `cf_sample_rank` (`Algorithm21B`)


In [ ]:
def reverse_lattice_step(*, l_t, pred_l, t_graph, num_atoms, dt):
    return runner.model.diffusion_l.reverse_step(
        t=t_graph.reshape(-1),
        x_t=l_t,
        pred=pred_l,
        dt=dt,
    )


def prepare_short_sampler_state(case: GraphCase, *, init_mode: str, t_start: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    formula = cf_formula_for_case(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, _ = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    start_times = make_times(batch_view, float(t_start))
    set_seed(int(seed) + 880000 + int(case.graph_id))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=start_times.nodes, f0=f0_model, index=batch_view.batch)
    l_t = case.gt_l_feature.detach().clone().reshape(1, -1)
    return {
        'payload': payload,
        'q0': q0,
        'batch': batch_view,
        'f_t': f_t.detach().clone(),
        'v_t': v_t.detach().clone(),
        'l_t': l_t,
        'a_t': case.atomic_numbers.detach().clone(),
        'node_index': batch_view.batch.detach().clone(),
        'edge_node_index': batch_view.edge_node_index.detach().clone(),
        'sampling_tdm': runner.model.tdm,
        'sampling_diffusion_l': runner.model.diffusion_l,
        'sampling_time_grid': np.linspace(float(t_start), 1e-6, int(SHORT_SAMPLER_STEPS) + 1, dtype=float),
    }


def iter_manual_sampling_times(state_dict):
    from kldmPlus.utils.time import make_times
    grid = np.asarray(state_dict['sampling_time_grid'], dtype=float)
    batch_view = state_dict['batch']
    for i in range(len(grid) - 1):
        t_now = float(grid[i])
        t_next = float(grid[i + 1])
        now = make_times(batch_view, t_now)
        nxt = make_times(batch_view, t_next)
        yield i + 1, SimpleNamespace(now=now, next=nxt, dt=float(t_now - t_next), t_next_float=float(t_next))


def run_short_sampler(case: GraphCase, *, mode: str, beta: float, seed: int = SAMPLE_SEED):
    notebook_log(f'[test13] short-sampler graph={case.graph_id} mode={mode} beta={beta} prepare start')
    print(f'[short-sampler] graph={case.graph_id} mode={mode} beta={beta} prepare start')
    prepared = prepare_short_sampler_state(case, init_mode='crystalformer_oracle', t_start=float(SHORT_SAMPLER_TSTART), seed=seed)
    payload = prepared['payload']
    formula = cf_formula_for_case(case)
    q_live = prepared['q0'].detach().clone()
    started = time.perf_counter()
    trace = []
    notebook_log(f'[test13] short-sampler graph={case.graph_id} mode={mode} prepared')
    print(f'[short-sampler] graph={case.graph_id} mode={mode} prepared f={tuple(prepared["f_t"].shape)} v={tuple(prepared["v_t"].shape)} l={tuple(prepared["l_t"].shape)} a={tuple(prepared["a_t"].shape)}')
    for step_idx, times in iter_manual_sampling_times(prepared):
        print(f'  [short-sampler] graph={case.graph_id} mode={mode} step={step_idx} t_now={float(times.now.graph.reshape(-1)[0]):.6f} t_next={float(times.t_next_float):.6f} start')
        try:
            with torch.no_grad():
                if step_idx == 1:
                    print(f'    [short-sampler] step=1 input shapes pos={tuple(prepared["f_t"].shape)} v={tuple(prepared["v_t"].shape)} l={tuple(prepared["l_t"].shape)} t_graph={tuple(times.now.graph.shape)} t_nodes={tuple(times.now.nodes.shape)}')
                print(f'    [short-sampler] step={step_idx} score_network start')
                preds_curr = runner.model.score_network(
                    t=times.now.graph,
                    pos=prepared['f_t'],
                    v=prepared['v_t'],
                    h=prepared['a_t'],
                    l=prepared['l_t'],
                    node_index=prepared['node_index'],
                    edge_node_index=prepared['edge_node_index'],
                )
                print(f'    [short-sampler] step={step_idx} score_network done pred_v={tuple(preds_curr["v"].shape)} pred_l={tuple(preds_curr["l"].shape)}')
                print(f'    [short-sampler] step={step_idx} reconstruct score_v start')
                score_v = prepared['sampling_tdm'].reconstruct_full_reverse_velocity_score(
                    t=times.now.nodes,
                    v_t=prepared['v_t'],
                    pred_v=preds_curr['v'],
                    index=prepared['node_index'],
                )
                print(f'    [short-sampler] step={step_idx} reconstruct score_v done score_v={tuple(score_v.shape)}')
                print(f'    [short-sampler] step={step_idx} reverse_exp_step start')
                prepared['f_t'], prepared['v_t'] = prepared['sampling_tdm'].reverse_exp_step(
                    f_t=prepared['f_t'],
                    v_t=prepared['v_t'],
                    score_v=score_v,
                    index=prepared['node_index'],
                    dt=times.dt,
                )
                print(f'    [short-sampler] step={step_idx} reverse_exp_step done f={tuple(prepared["f_t"].shape)} v={tuple(prepared["v_t"].shape)}')
                print(f'    [short-sampler] step={step_idx} reverse_lattice_step start l_t={tuple(prepared["l_t"].shape)} pred_l={tuple(preds_curr["l"].shape)} t_graph={tuple(times.now.graph.reshape(-1).shape)} num_atoms={tuple(prepared["batch"].num_atoms.shape)} dt={times.dt}')
                prepared['l_t'] = reverse_lattice_step(
                    l_t=prepared['l_t'],
                    pred_l=preds_curr['l'],
                    t_graph=times.now.graph,
                    num_atoms=prepared['batch'].num_atoms,
                    dt=times.dt,
                )
                print(f'    [short-sampler] step={step_idx} reverse_lattice_step done l={tuple(prepared["l_t"].shape)}')
        except Exception as exc:
            print(f'    [short-sampler] step={step_idx} FAILED during reverse block: {type(exc).__name__}: {exc}')
            raise
        should_project = (mode != 'baseline' and algorithm21_should_project(float(times.t_next_float), config21(beta=float(beta), alpha=0.25, debug_prints=(mode == 'cf_local_rerank'))))
        print(f'    [short-sampler] step={step_idx} should_project={should_project}')
        if should_project:
            try:
                print(f'    [short-sampler] step={step_idx} make_algo_state start l={tuple(prepared["l_t"].shape)}')
                algo_state = make_algo_state_from_raw(
                    f=prepared['f_t'],
                    v=prepared['v_t'],
                    l=prepared['l_t'],
                    atom_types=prepared['a_t'],
                    node_index=prepared['node_index'],
                    edge_node_index=prepared['edge_node_index'],
                    t_graph=times.next.graph,
                    t_nodes=times.next.nodes,
                )
                print(f'    [short-sampler] step={step_idx} project_renoise start')
                if mode == 'cf_sample_rank':
                    step = algorithm21_project_renoise_step_local_rerank(
                        state=algo_state,
                        payload=payload,
                        model=runner.model,
                        alpha=0.25,
                        cf_likelihood=cf_like_runtime,
                        formula=formula,
                    )
                    if ranked_items:
                        q_live = ranked_items[0].q.detach().clone()
                else:
                    step = algorithm21_project_renoise_step(
                        state=algo_state,
                        payload=payload,
                        model=runner.model,
                        config=config21(beta=float(beta), alpha=0.25),
                        cf_likelihood=None,
                        formula=formula,
                        q_init=q_live,
                    )
                print(f'    [short-sampler] step={step_idx} project_renoise done accepted={bool(step.accepted)} witness_before={float(step.fit_before.witness_sin):.6g} witness_after={float(step.fit_after.witness_sin):.6g}')
                prepared['f_t'] = step.state_after.f.detach().clone()
                prepared['v_t'] = step.state_after.v.detach().clone()
                q_live = step.fit_after.q_star.detach().clone()
                trace.append({
                    'graph': case.graph_id,
                    'mode': mode,
                    'beta': float(beta),
                    'step': int(step_idx),
                    't_next': float(times.t_next_float),
                    'accepted': bool(step.accepted),
                    'score_before': float(step.fit_before.score_total),
                    'score_after': float(step.fit_after.score_total),
                    'witness_before': float(step.fit_before.witness_sin),
                    'witness_after': float(step.fit_after.witness_sin),
                })
            except Exception as exc:
                print(f'    [short-sampler] step={step_idx} FAILED during projection block: {type(exc).__name__}: {exc}')
                raise
    print(f'[short-sampler] graph={case.graph_id} mode={mode} final_state start')
    final_state = make_algo_state_from_raw(
        f=prepared['f_t'],
        v=prepared['v_t'],
        l=prepared['l_t'],
        atom_types=prepared['a_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
        t_graph=torch.as_tensor([[1e-6]], device=prepared['f_t'].device, dtype=prepared['f_t'].dtype),
        t_nodes=torch.full((int(prepared['f_t'].shape[0]),), 1e-6, device=prepared['f_t'].device, dtype=prepared['f_t'].dtype),
    )
    print(f'[short-sampler] graph={case.graph_id} mode={mode} final_clean start')
    final_clean = clean_prediction(final_state)
    print(f'[short-sampler] graph={case.graph_id} mode={mode} final_fit start')
    final_fit = fit_clean_q(final_clean, case=case, payload=payload, config=config21(beta=0.0, alpha=0.25, q_opt_steps=Q_OPT_STEPS), cf_like=None, formula=formula, q_init=q_live)
    final_e_cf = cf_nll_eval(payload=payload, q=final_fit.q_star, lattice_feature=prepared['l_t'].reshape(-1), formula=formula, label=f'test13_graph{case.graph_id}_{mode}_final_q') if cf_like_runtime is not None else float('nan')
    print(f'[short-sampler] graph={case.graph_id} mode={mode} evaluate_arrays start')
    endpoint = evaluate_arrays(case, final_clean, prepared['l_t'].reshape(-1), prepared['a_t'])
    print(f'[short-sampler] graph={case.graph_id} mode={mode} done final_witness={float(final_fit.witness_sin):.6g} frac_rmse={float(endpoint["frac_rmse"]):.6g} E_CF={float(final_e_cf):.6g}')
    return {
        'test': 'algorithm21_short_sampler',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'mode': mode,
        'beta': float(beta),
        'runtime_s': float(time.perf_counter() - started),
        'accepted_fraction': float(np.mean([row['accepted'] for row in trace])) if trace else np.nan,
        'final_witness': float(final_fit.witness_sin),
        'final_E_CF': float(final_e_cf),
        'frac_rmse': float(endpoint['frac_rmse']),
        'rmse': float(endpoint['rmse']) if endpoint['rmse'] is not None else float('nan'),
        'match': bool(endpoint['match']),
        'valid': bool(endpoint['valid']),
        'PASS': True,
        'status': 'INFO',
    }, trace

rows = []
trace_rows = []
cf_like_runtime = globals().get('cf_like', None)
job_specs = [('cf_sample_rank', 0.0), ('beta0_q_projection', 0.0), ('baseline', 0.0)]
print(f'algorithm21 short sampler graphs={[case.graph_id for case in GRAPH_CASES]} t_start={SHORT_SAMPLER_TSTART} n_steps={SHORT_SAMPLER_STEPS} jobs={job_specs}')
for case in GRAPH_CASES:
    for mode, beta in job_specs:
        try:
            if mode == 'cf_sample_rank' and cf_like_runtime is None:
                raise RuntimeError('CrystalFormer checkpoint not available for CrystalFormer-guided run.')
            row, trace = run_short_sampler(case, mode=mode, beta=float(beta))
            rows.append(row)
            trace_rows.extend(trace)
        except Exception as exc:
            rows.append(error_row('algorithm21_short_sampler', case.graph_id, exc, 'short_sampler', space_group=case.requested_sg, mode=mode, beta=float(beta)))

compare_df = dataframe_result('algorithm21_short_sampler', rows)
safe_display_sorted(compare_df, ['graph','mode','beta'])
trace_df = dataframe_result('algorithm21_short_sampler_trace', trace_rows)
safe_display_sorted(trace_df, ['graph','mode','step'])

notebook_log('[test13] cleanup start')
notebook_cleanup()
notebook_log('[test13] cleanup done')


## Tutorial Part C — Template Proposal and Template-q Ranking Diagnostics

This section keeps the oracle space group but removes the oracle template. It uses PyXtal template enumeration, synthetic payload materialization, and species-matched witness scoring against the KLDM clean prediction.


In [ ]:
rows_c = []
tutorial_c_t = float(TEST46_CORE_T_VALUES[0]) if len(TEST46_CORE_T_VALUES) else 0.5
K_template = 32
K_q = 8
notebook_log(f'[tutorial-c] start graphs={[case.graph_id for case in GRAPH_CASES]} K_template={K_template} K_q={K_q} t={tutorial_c_t}')
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    formula = cf_formula_for_case(case)
    state = sample_gt_noisy_state(case, t_value=tutorial_c_t)
    f0_hat = clean_prediction(state)
    z_hat = map_model_to_payload_reference_chart(f0_hat, payload)
    oracle_signature = tuple(sorted((int(z), str(label)) for z, label in zip(payload.anchor_atomic_numbers.tolist(), payload.wyckoff_letters)))
    templates = extract_wyckoff_templates(space_group_number=case.requested_sg, atomic_numbers=case.atomic_numbers, max_templates=K_template)
    valid_templates = 0
    candidate_rows = []
    oracle_rank = None
    for template_rank, template in enumerate(templates):
        signature = flatten_site_signature(template)
        species_counts_ok = sorted([int(site.atomic_number) for site in template.site_templates for _ in range(int(site.multiplicity))]) == sorted(case.atomic_numbers.detach().cpu().tolist())
        multiplicity_ok = int(template.total_atoms) == int(case.gt_frac.shape[0])
        if species_counts_ok and multiplicity_ok:
            valid_templates += 1
        if signature == oracle_signature and oracle_rank is None:
            oracle_rank = int(template_rank)
        try:
            if signature == oracle_signature:
                q_seed = recover_template_free_vars_from_anchor_entries(template, payload.anchor_entries).to(case.gt_frac.device, dtype=case.gt_frac.dtype)
            else:
                q_seed = sample_random_free_vars(template, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
            synth_payload = build_payload_from_template_q(template=template, q=q_seed, lattice_matrix=payload.lattice_matrix)
            template_qs = [q_seed]
            template_cf = [float('nan')]
            if cf_like is not None and template.total_free_dims >= 0:
                try:
                    q_more, cf_more = sample_q_from_crystalformer(payload=synth_payload, lattice_feature=case.gt_l_feature, formula=formula, cf_likelihood=cf_like, K=K_q, top_p=1.0, temperature=1.0, seed=SAMPLE_SEED + int(case.graph_id) * 1000 + int(template_rank))
                    if q_more:
                        template_qs = q_more
                        template_cf = cf_more
                except Exception:
                    pass
            best_witness = float('inf')
            best_post = float('inf')
            for q_idx, q_try in enumerate(template_qs):
                synth_try = build_payload_from_template_q(template=template, q=q_try, lattice_matrix=payload.lattice_matrix)
                aligned = species_match_reorder(source_frac=synth_try.expanded_frac_coords, source_atomic_numbers=synth_try.expanded_atomic_numbers, target_frac=z_hat, target_atomic_numbers=payload.expanded_atomic_numbers)
                z_hard_aligned = torch.as_tensor(aligned['aligned_source_in_target_order'], device=z_hat.device, dtype=z_hat.dtype)
                witness = float(algo21_backend.witness_torus_sin_loss(z_hard_aligned, z_hat).detach().item())
                best_witness = min(best_witness, witness)
                if q_idx < 3:
                    step = algorithm21_project_renoise_step_from_q(state=state, payload=payload, model=runner.model, q_star=q_try.to(case.gt_frac.device, dtype=case.gt_frac.dtype), alpha=0.25, post_renoise_accept=True) if signature == oracle_signature else None
                    if step is not None:
                        best_post = min(best_post, float(step.fit_after.witness_sin))
            candidate_rows.append((best_witness, template_rank, signature, best_post))
        except Exception as exc:
            candidate_rows.append((float('inf'), template_rank, signature, float('inf')))
    candidate_rows.sort(key=lambda item: item[0])
    top3 = candidate_rows[:3]
    rows_c.append({
        'test': 'algorithm21c_template_q_diagnostic',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'num_templates': int(len(templates)),
        'valid_templates': int(valid_templates),
        'oracle_template_in_topK': bool(oracle_rank is not None),
        'oracle_template_rank_by_generator': int(oracle_rank) if oracle_rank is not None else -1,
        'top1_witness': float(top3[0][0]) if top3 else float('nan'),
        'top3_templates': [str(item[2]) for item in top3],
        'top3_post_renoise_min_witness': float(min([item[3] for item in top3])) if top3 else float('nan'),
        'PASS': bool(valid_templates > 0),
        'status': 'INFO',
    })
    notebook_cleanup()
display(dataframe_result('algorithm21c_template_q_diagnostic', rows_c).sort_values(['graph']).reset_index(drop=True))


## Part C Intermediate — Template Generator Feasibility

Split the Version C question into smaller checks:

- how many PyXtal templates are generated?
- how many match stoichiometry and total multiplicity?
- can synthetic payloads be materialized from those templates?


In [ ]:
rows_c_mid = []
for case in GRAPH_CASES:
    payload = build_oracle_payload(case)
    templates = extract_wyckoff_templates(space_group_number=case.requested_sg, atomic_numbers=case.atomic_numbers, max_templates=32)
    materialized = 0
    valid_count = 0
    for template in templates:
        species_counts_ok = sorted([int(site.atomic_number) for site in template.site_templates for _ in range(int(site.multiplicity))]) == sorted(case.atomic_numbers.detach().cpu().tolist())
        multiplicity_ok = int(template.total_atoms) == int(case.gt_frac.shape[0])
        if species_counts_ok and multiplicity_ok:
            valid_count += 1
        try:
            q0 = sample_random_free_vars(template, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
            _ = build_payload_from_template_q(template=template, q=q0, lattice_matrix=payload.lattice_matrix)
            materialized += 1
        except Exception:
            pass
    rows_c_mid.append({
        'test': 'algorithm21c_intermediate_template_generator',
        'graph': case.graph_id,
        'space_group': case.requested_sg,
        'num_templates_total': int(len(templates)),
        'num_templates_valid': int(valid_count),
        'num_templates_materialized': int(materialized),
        'PASS': bool(materialized > 0),
        'status': 'INFO',
    })
display(dataframe_result('algorithm21c_intermediate_template_generator', rows_c_mid).sort_values(['graph']).reset_index(drop=True))


## Final Audit Summary

Collect the focused Algorithm21 pipeline tables only.


In [ ]:
focused_keys = [
    'algorithm21_cf_checkpoint_load',
    'algorithm21a_a0_frame_denoiser_feasibility',
    'algorithm21a_a1_payload_chart_sanity',
    'algorithm21a_a2_q_projection',
    'algorithm21a_a3_q_init_ablation',
    'algorithm21a_a4_a5_alpha_renoise',
    'algorithm21a_intermediate_fit_anchor',
    'algorithm21b_cf_sample_quality',
    'algorithm21b_top3_branch',
    'algorithm21b_intermediate_candidate_branch',
    'algorithm21_short_sampler',
    'algorithm21c_template_q_diagnostic',
    'algorithm21c_intermediate_template_generator',
]
summary_rows = []
for key in focused_keys:
    df = result_tables.get(key)
    if df is None:
        summary_rows.append({'table': key, 'rows': 0, 'pass_rate': np.nan})
        continue
    pass_rate = float(df['PASS'].mean()) if 'PASS' in df.columns and len(df) else np.nan
    summary_rows.append({'table': key, 'rows': int(len(df)), 'pass_rate': pass_rate})
display(pd.DataFrame(summary_rows))
